In [ ]:
## Begin with some analysis
import pandas as pd

data_file = "./ehr_data/diagnosis.csv"

print("Loading diagnosis data...")
df_diag = pd.read_csv(data_file, dtype=str)

print(f"Loaded {len(df_diag)} rows.")
print(df_diag.columns)

print("\nTop 20 most common diagnosis names:")
print(df_diag['DX_NAME'].value_counts().head(20))

print("\nTop 20 most common diagnosis codes:")
print(df_diag['DX_CODE'].value_counts().head(20))

keywords = {
    'sleep_apnea': ['sleep apnea', 'obstructive sleep apnea'],
    'asthma': ['asthma'],
    'obesity': ['obesity'],
    'diabetes': ['diabetes'],
    'hypertension': ['hypertension'],
    'depression': ['depression', 'mood disorder'],
    'anxiety': ['anxiety'],
    'adhd': ['adhd'],
}

def match_any(text, terms):
    text = str(text).lower()
    return any(t in text for t in terms)

for label, terms in keywords.items():
    df_diag[label] = df_diag['DX_NAME'].apply(lambda x: int(match_any(x, terms)))

print("\nComorbidity counts:")
print(df_diag[list(keywords.keys())].sum().sort_values(ascending=False))

In [ ]:
import pandas as pd

demo_path = "./ehr_data/demographic.csv"
df = pd.read_csv(demo_path, dtype=str)

df['RACE_DESCR'] = df['RACE_DESCR'].fillna('UNKNOWN').str.strip().str.upper()
race_counts = df['RACE_DESCR'].value_counts(dropna=False)

print("Unique race descriptions:\n")
print(race_counts)

In [ ]:
import pandas as pd
import numpy as np
import os
from glob import glob
from collections import defaultdict

data_dir = "./ehr_data"
out_dir = "./output_embeddings"
os.makedirs(out_dir, exist_ok=True)

keywords = {
    'asthma': ['asthma'],
    'obesity': ['obesity'],
    'diabetes': ['diabetes'],
    'hypertension': ['hypertension'],
    'anxiety': ['anxiety'],
    'depression': ['depression', 'mood disorder'],
    'adhd': ['adhd'],
    'seizure': ['seizure', 'epilepsy'],
    'gerd': ['reflux', 'gerd', 'gastroesophageal'],
    'cerebral_palsy': ['cerebral palsy'],
    'autism': ['autism'],
    'dev_delay': ['developmental delay', 'speech delay', 'language delay'],
}
keyword_names = list(keywords.keys())

print("Loading files...")
diag = pd.read_csv(os.path.join(data_dir, "diagnosis.csv"), dtype=str)
demo = pd.read_csv(os.path.join(data_dir, "demographic.csv"), dtype=str)
study = pd.read_csv(os.path.join(data_dir, "sleep_study.csv"), dtype=str)
study["AGE_AT_SLEEP_STUDY_DAYS"] = pd.to_numeric(study["AGE_AT_SLEEP_STUDY_DAYS"], errors='coerce')
demo = demo.set_index("STUDY_PAT_ID")

print("Checking existing embeddings...")
existing_pairs = set()
for file in glob(os.path.join(out_dir, "*_embeddings.npy")):
    base = os.path.basename(file).replace("_embeddings.npy", "")
    existing_pairs.add(base)
print(f"Found {len(existing_pairs)} embedding pairs.")

print("Extracting comorbidity flags...")
diag['DX_NAME'] = diag['DX_NAME'].fillna("").str.lower()
patient_flags = defaultdict(lambda: [0] * len(keyword_names))
for _, row in diag.iterrows():
    pid = row['STUDY_PAT_ID']
    dx_name = row['DX_NAME']
    for j, name in enumerate(keyword_names):
        if any(k in dx_name for k in keywords[name]):
            patient_flags[pid][j] = 1

gender_map = {"F": 0, "M": 1}
race_list = ["WHITE", "BLACK", "ASIAN", "OTHER", "MULTIRACIAL", "UNKNOWN"]
race_map = {
    "WHITE": "WHITE",
    "BLACK OR AFRICAN AMERICAN": "BLACK",
    "ASIAN": "ASIAN",
    "MULTIPLE RACE": "MULTIRACIAL",
    "REFUSE TO ANSWER": "UNKNOWN",
    "UNKNOWN": "UNKNOWN",
    "NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER": "OTHER",
    "AMERICAN INDIAN OR ALASKA NATIVE": "OTHER",
}

def build_ehr_vector(pid, sid):
    eid = f"{pid}_{sid}"
    if eid not in existing_pairs:
        return None
    row = study[(study["STUDY_PAT_ID"] == pid) & (study["SLEEP_STUDY_ID"] == sid)]
    if row.empty:
        return None
    age = row["AGE_AT_SLEEP_STUDY_DAYS"].values[0]
    age = float(age) if not pd.isnull(age) else np.nan

    if pid in demo.index:
        gender = demo.loc[pid, "PCORI_GENDER_CD"].upper()
        gender_onehot = [0, 0, 0]
        gender_onehot[gender_map.get(gender, 2)] = 1
        race_raw = str(demo.loc[pid, "RACE_DESCR"]).strip().upper()
        race_clean = race_map.get(race_raw, "UNKNOWN")
        race_onehot = [int(race_clean == r) for r in race_list]
        hispanic = int(str(demo.loc[pid, "PCORI_HISPANIC_CD"]).upper() == "Y")
    else:
        gender_onehot = [0, 0, 1]
        race_onehot = [0] * len(race_list)
        hispanic = 0

    comorbs = patient_flags.get(pid, [0] * len(keyword_names))
    return np.array([age] + gender_onehot + race_onehot + [hispanic] + comorbs, dtype=np.float32)

pairs = study[["STUDY_PAT_ID", "SLEEP_STUDY_ID"]].dropna().drop_duplicates()
count = 0

print("Building and saving EHR features...")
for _, (pid, sid) in pairs.iterrows():
    eid = f"{pid}_{sid}"
    vec = build_ehr_vector(pid, sid)
    if vec is not None:
        outpath = os.path.join(out_dir, f"{eid}_ehr_feature.npy")
        np.save(outpath, vec)
        count += 1

print(f"Saved {count} EHR feature files.")

In [ ]:
import numpy as np
import pandas as pd
import os
from glob import glob
import matplotlib.pyplot as plt

feature_dir = "./output_embeddings"
ehr_files = sorted(glob(os.path.join(feature_dir, "*_ehr_feature.npy")))

gender_labels = ["Female", "Male", "Other"]
race_labels = ["White", "Black", "Asian", "Other", "Multiracial", "Unknown"]
comorbidity_labels = [
    'asthma', 'obesity', 'diabetes', 'hypertension', 'anxiety',
    'depression', 'adhd', 'seizure', 'gerd', 'cerebral_palsy', 'autism', 'dev_delay'
]

n_features = 1 + len(gender_labels) + len(race_labels) + 1 + len(comorbidity_labels)

print(f"Loading {len(ehr_files)} vectors...")
data, ids = [], []

for f in ehr_files:
    vec = np.load(f)
    if vec.shape[0] == n_features:
        data.append(vec)
        ids.append(os.path.basename(f).replace("_ehr_feature.npy", ""))
    else:
        print(f"Skipped {f}, wrong shape {vec.shape}")

df = pd.DataFrame(
    data, index=ids,
    columns=["age_days"] + gender_labels + race_labels + ["hispanic"] + comorbidity_labels
)

print("Dataframe shape:", df.shape)

print("\nSummary Statistics:")
print(df.describe(include='all'))

plt.figure()
df['age_days'].hist(bins=30)
plt.title("Age Distribution")
plt.xlabel("Age (days)")
plt.ylabel("Count")
plt.grid(True)
plt.tight_layout()
plt.show()

df_gender = df[gender_labels].sum().sort_values(ascending=False)
df_gender.plot(kind='bar', title="Gender Distribution", figsize=(6, 4))
plt.ylabel("Count")
plt.tight_layout()
plt.show()

df_race = df[race_labels].sum().sort_values(ascending=False)
df_race.plot(kind='bar', title="Race Distribution", figsize=(6, 4))
plt.ylabel("Count")
plt.tight_layout()
plt.show()

df_comorb = df[comorbidity_labels].sum().sort_values(ascending=False)
df_comorb.plot(kind='barh', title="Comorbidity Prevalence", figsize=(7, 6))
plt.xlabel("Count")
plt.tight_layout()
plt.show()